## Generate Catchment Pourpoints

In [ ]:
import os,sys,psycopg2,tempfile,datetime;
from contextlib import closing;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

dbse = os.environ['POSTGRESQL_DB'];
host = os.environ['POSTGRESQL_HOST'];
port = os.environ['POSTGRESQL_PORT'];
user = 'cipsrv';
pswd = os.environ['POSTGRESQL_CIP_PASS'];


In [ ]:
src_cat = 'cipstg_nhdplus_h_r2.nhdpluscatchment';
trg_fpp = 'cipstg_nhdplus_h_r2.nhdpluscatchmentfpp';
trg_tpp = 'cipstg_nhdplus_h_r2.nhdpluscatchmenttpp';
trg_vpp = 'cipstg_nhdplus_h_r2.nhdpluscatchmentvpp';
fdr_sch = 'cipsrv_nhdplus_h';
wrk_sch = 'cipstg_nhdplus_h_r2';


In [ ]:
cs = "dbname=%s user=%s password=%s host=%s port=%s" % (
     dbse
    ,user
    ,pswd
    ,host
    ,port
);

try:
    conn = psycopg2.connect(cs);
    conn.close();
except:
    raise Exception("database connection error");

print("database is ready");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time
conn = psycopg2.connect(cs);
with closing(conn.cursor()) as cursor:
    cursor.execute("""
CREATE OR REPLACE FUNCTION """ + wrk_sch + """.fdr_pourpoint(
    pAreaOfInterest       IN  GEOMETRY
   ,pVPPCellSearch        IN  INTEGER DEFAULT 4
   ,pKnownRegion          IN  VARCHAR DEFAULT NULL
   ,pFP                   OUT GEOMETRY 
   ,pTP                   OUT GEOMETRY
   ,pVPP                  OUT GEOMETRY
   ,pPPType               OUT VARCHAR
   ,pFPAccumulation       OUT INTEGER
   ,pFPAccumulationFDR    OUT INTEGER
   ,pFPAccumlationSink    OUT BOOLEAN
   ,pVPPAccumulation      OUT INTEGER
   ,pReturnCode           OUT INTEGER
   ,pStatusMessage        OUT VARCHAR
)
STABLE
AS
$BODY$ 
DECLARE
   r                   RECORD;
   int_vpp_cell_search INTEGER := pVPPCellSearch;
   rast_accum          RASTER;
   rast_work           RASTER;
   int_raster_srid     INTEGER;
   int_value           INTEGER;
   geom_fp             GEOMETRY;
   geom_tp             GEOMETRY;
   geom_vpp            GEOMETRY;
   int_accum_columnx   INTEGER;
   int_accum_rowy      INTEGER;
   int_work_columnx    INTEGER;
   int_work_rowy       INTEGER;
   boo_has_tp          BOOLEAN;
   int_largest         INTEGER;
   mat_accum           INTEGER[][];
   int_work_val        INTEGER;
   
BEGIN

   ----------------------------------------------------------------------------
   -- Step 10
   -- Check over incoming parameters
   ----------------------------------------------------------------------------
   pReturnCode := 0;
   boo_has_tp  := TRUE;
   
   IF pAreaOfInterest IS NULL
   THEN
      pReturnCode    := -1;
      pStatusMessage := 'input area of interest is null';
      RETURN;
      
   END IF;
   
   IF int_vpp_cell_search IS NULL
   OR int_vpp_cell_search < 1
   THEN
      int_vpp_cell_search := 4;
      
   END IF;
   
   ----------------------------------------------------------------------------
   -- Step 20
   -- Get the from pourpoint for the area of interest
   ----------------------------------------------------------------------------
   r := """ + fdr_sch + """.fdr_flowaccumulation(
       p_area_of_interest := pAreaOfInterest
      ,p_default_weight   := 1
   );
   rast_accum            := r.out_flow_accumulation;
   pFPAccumulation       := r.out_max_accumulation;
   int_accum_columnx     := r.out_max_accumulation_x;
   int_accum_rowy        := r.out_max_accumulation_y;
   int_raster_srid       := r.out_raster_srid;
   pReturnCode           := r.out_return_code;
   pStatusMessage        := r.out_status_message;

   IF pReturnCode != 0
   THEN
      RETURN;

   END IF;

   ----------------------------------------------------------------------------
   -- Step 30
   -- Determine the from pour point geometry
   ----------------------------------------------------------------------------
   geom_fp := r.out_max_accumulation_pt;
   
   IF geom_fp IS NULL
   THEN
      pReturnCode    := -1;
      pStatusMessage := 'unable to determine maximum accumulation';
      RETURN;
      
   END IF;

   ----------------------------------------------------------------------------
   -- Step 40
   -- Fetch the flow direction grid around the from pour point
   ----------------------------------------------------------------------------
   r := """ + fdr_sch + """.fetch_grids_by_geometry(
       p_FDR_input         := public.ST_Expand(geom_fp,100)
      ,p_FAC_input         := NULL
      ,p_known_region      := int_raster_srid::VARCHAR
      ,p_FDR_nodata        := 255
      ,p_FAC_nodata        := NULL
      ,p_crop              := FALSE
   );

   IF rast_work IS NULL
   THEN
      RAISE WARNING 'unable to inspect area around potential pourpoint';
      pFP  := NULL;
      pTP  := NULL;
      pVPP := NULL;
      RETURN;

   END IF;

   ----------------------------------------------------------------------------
   -- Step 50
   -- Get the value 
   ----------------------------------------------------------------------------
   r := public.ST_WorldToRasterCoord(
       rast   := rast_work
      ,pt     := geom_fp
   );
   int_work_columnx := r.columnx;
   int_work_rowy    := r.rowy;

   pFPAccumulationFDR := public.ST_Value(
       rast := rast_work
      ,x := int_work_columnx
      ,y := int_work_rowy
   );
   
   ----------------------------------------------------------------------------
   -- Step 60
   -- Walk the grid
   ----------------------------------------------------------------------------
   pPPType := 'Outflow';
   pFPAccumlationSink := FALSE;
   
   CASE pFPAccumulationFDR
   WHEN 1
   THEN
      int_work_columnx := int_work_columnx + 1;
      
   WHEN 2
   THEN
      int_work_columnx := int_work_columnx + 1;
      int_work_rowy    := int_work_rowy    + 1;
      
   WHEN 4
   THEN
      int_work_rowy    := int_work_rowy    + 1;
      
   WHEN 8
   THEN
      int_work_columnx := int_work_columnx - 1;
      int_work_rowy    := int_work_rowy    + 1;
      
   WHEN 16
   THEN
      int_work_columnx := int_work_columnx - 1;
      
   WHEN 32
   THEN
      int_work_columnx := int_work_columnx - 1;
      int_work_rowy    := int_work_rowy    - 1;
      
   WHEN 64
   THEN
      int_work_rowy    := int_work_rowy    - 1;
      
   WHEN 128
   THEN
      int_work_columnx := int_work_columnx + 1;
      int_work_rowy    := int_work_rowy    - 1;
      
   WHEN 0
   THEN
      pPPType    := 'Sink';
      boo_has_tp := FALSE;
      pFPAccumlationSink := TRUE;
      
   WHEN 255
   THEN
      pPPType    := 'GridExit';
      boo_has_tp := FALSE;
      
   ELSE
      pReturnCode    := -1;
      pStatusMessage := 'error in flow direction grid, value ' || pFPAccumulationFDR::VARCHAR;
      RETURN;
      
   END CASE;
   
   ----------------------------------------------------------------------------
   -- Step 70
   -- Generate tp if exists
   ----------------------------------------------------------------------------
   IF boo_has_tp
   THEN
      geom_tp := public.ST_PixelAsCentroid(
          rast   := rast_work
         ,x      := int_work_columnx
         ,y      := int_work_rowy
      );
   
   END IF;
   
   ----------------------------------------------------------------------------
   -- Step 80
   -- Determine vector pour point from raster accumulation grid
   ----------------------------------------------------------------------------
   mat_accum := public.ST_Neighborhood(
       rast      := rast_accum
      ,band      := 1
      ,columnx   := int_accum_columnx
      ,rowy      := int_accum_rowy
      ,distancex := int_vpp_cell_search
      ,distancey := int_vpp_cell_search
      ,exclude_nodata_value := TRUE
   );

   pVPPAccumulation := 0;
   FOR i IN 1 .. array_length(mat_accum,1)
   LOOP
      FOR j IN 1 .. array_length(mat_accum,2)
      LOOP
         IF i = 1 OR i = array_length(mat_accum,1)
         OR j = 1 OR j = array_length(mat_accum,2)
         THEN      
            IF mat_accum[j][i] > pVPPAccumulation
            THEN
               pVPPAccumulation := mat_accum[j][i];
               int_work_columnx := (int_accum_columnx - int_vpp_cell_search - 1) + i;
               int_work_rowy    := (int_accum_rowy    - int_vpp_cell_search - 1) + j;
               
            END IF;

         END IF;            
         
      END LOOP;
      
   END LOOP;
   
   geom_vpp := public.ST_PixelAsCentroid(
       rast   := rast_accum
      ,x      := int_work_columnx
      ,y      := int_work_rowy
   );

   ----------------------------------------------------------------------------
   -- Step 90
   -- Project results to 4269
   ----------------------------------------------------------------------------
   pFP := public.ST_Transform(geom_fp,4269);
   
   IF geom_tp IS NOT NULL
   THEN
      pTP := public.ST_Transform(geom_tp,4269);
      
   END IF;
   
   pVPP := public.ST_Transform(geom_vpp,4269);

END;
$BODY$
LANGUAGE plpgsql;
    """);

    cursor.execute("""
GRANT EXECUTE ON FUNCTION """ + wrk_sch + """.fdr_pourpoint(
    GEOMETRY
   ,INTEGER
   ,VARCHAR
) TO """ + user + """;
    """);
    
    conn.commit();

conn.close();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");
print("pp function setup.");

In [ ]:
%%time
purge_data = False;

if purge_data is True:
    conn = psycopg2.connect(cs);
    with closing(conn.cursor()) as cursor:
        cursor.execute("TRUNCATE TABLE " + trg_fpp);
        cursor.execute("TRUNCATE TABLE " + trg_tpp);
        cursor.execute("TRUNCATE TABLE " + trg_vpp);
        
        conn.commit();
    
    conn.close();
    
    print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");
    print("any preexisting data purged.");
    

In [ ]:
sql = """
DO $$DECLARE
    rec                 RECORD;
    rec2                RECORD;
BEGIN

    FOR rec IN 
    SELECT
     a.objectid
    ,a.nhdplusid
    ,a.sourcefc
    ,a.vpuid
    ,a.shape
    FROM
    """ + src_cat + """ a
    WHERE
    /* SUBSTR(a.vpuid,1,2) IN ('01') */
    a.vpuid IN ('0102')
    LOOP   
        rec2 := """ + wrk_sch + """.fdr_pourpoint(
             pAreaOfInterest       := rec.shape
            ,pVPPCellSearch        := 4
            ,pKnownRegion          := NULL
        );

        IF rec2.pFP IS NOT NULL
        THEN
            INSERT INTO """ + trg_fpp + """(
                 objectid
                ,nhdplusid
                ,sourcefc
                ,vpuid
                ,pp_type
                ,accumulation
                ,accumulation_fdr
                ,accumulation_sink
                ,shape
            ) VALUES (
                 rec.objectid
                ,rec.nhdplusid
                ,rec.sourcefc
                ,rec.vpuid
                ,rec2.pPPType
                ,rec2.pFPAccumulation
                ,rec2.pFPAccumulationFDR
                ,rec2.pFPAccumlationSink
                ,rec2.pFP
            );

            IF rec2.pTP IS NOT NULL
            THEN
                INSERT INTO """ + trg_tpp + """(
                     objectid
                    ,nhdplusid
                    ,sourcefc
                    ,vpuid
                    ,pp_type
                    ,shape
                ) VALUES (
                     rec.objectid
                    ,rec.nhdplusid
                    ,rec.sourcefc
                    ,rec.vpuid
                    ,rec2.pPPType
                    ,rec2.pTP
                );

            END IF;

            IF rec2.pVPP IS NOT NULL
            AND rec2.pVPPAccumulation > 0
            THEN
                INSERT INTO """ + trg_vpp + """(
                     objectid
                    ,nhdplusid
                    ,sourcefc
                    ,vpuid
                    ,pp_type
                    ,accumulation
                    ,shape
                ) VALUES (
                     rec.objectid
                    ,rec.nhdplusid
                    ,rec.sourcefc
                    ,rec.vpuid
                    ,rec2.pPPType
                    ,rec2.pVPPAccumulation
                    ,rec2.pVPP
                );

            END IF;

        END IF;
   
   END LOOP;
   
END$$;
    """;

#print(sql);

In [ ]:
%%time
conn = psycopg2.connect(cs);
with closing(conn.cursor()) as cursor:
    cursor.execute(sql);
        
    conn.commit();

conn.close();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");